# Uno Feature Regression

Train a hybrid regression model from Uno feature values to aberration coefficients.

Required input: `outputs/uno_relationships/uno_coefficient_relationships.csv`, generated by `notebooks/uno_coefficient_relationship_short.ipynb` in the same Colab runtime or copied into the repo.

Model direction:

```text
feature values -> aberration coefficients
```

Complex quantities use real/imaginary components. The model is a calibrated linear inverse baseline plus a residual MLP.
Scalability note: this first version trains from the generated CSV table in memory. The model interface is intentionally feature-table based, so later large combinatorial datasets can be written in shards and loaded with a streaming `Dataset` without changing the feature/target definitions.

Results are saved first inside the Colab runtime under `/content/Aberration-Simulation/training_results/feature_regression`. The Google Drive cell is optional and only mirrors that folder when enabled.


## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Clone or Pull Latest GitHub Code

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/DrYGuo/Aberration-Simulation.git"
ROOT = Path("/content/Aberration-Simulation")

if ROOT.exists():
    print("Repository already exists. Pulling latest main...")
    subprocess.run(["git", "fetch", "origin", "main"], cwd=ROOT, check=True)
    subprocess.run(["git", "reset", "--hard", "origin/main"], cwd=ROOT, check=True)
else:
    print("Cloning repository...")
    subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)

os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "scripts"))

commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=ROOT, text=True).strip()
print("repo root:", ROOT)
print("commit:", commit)

## 3. Install and Verify Dependencies

In [ ]:
import importlib.util
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
if importlib.util.find_spec("torch") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch"], check=True)

print("Dependencies are ready.")

## 4. Locate Training CSV

In [ ]:
from pathlib import Path

CSV_PATH = ROOT / "outputs" / "uno_relationships" / "uno_coefficient_relationships.csv"
if not CSV_PATH.exists():
    raise FileNotFoundError(
        f"Missing {CSV_PATH}. Run notebooks/uno_coefficient_relationship_short.ipynb first "
        "in this Colab runtime, or copy the CSV into outputs/uno_relationships/."
    )
print("training CSV:", CSV_PATH)
print("size bytes:", CSV_PATH.stat().st_size)

## 5. Train Hybrid Feature Regressor

In [ ]:
from feature_regression_model import (
    FEATURE_COLUMNS,
    describe_hybrid_model,
    TARGET_COLUMNS,
    train_hybrid_regressor,
)

OUTPUT_DIR = ROOT / "training_results" / "feature_regression"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("feature columns:")
for name in FEATURE_COLUMNS:
    print(" ", name)
print("target columns:")
for name in TARGET_COLUMNS:
    print(" ", name)

print("model structure:")
print(describe_hybrid_model())

metrics = train_hybrid_regressor(
    CSV_PATH,
    OUTPUT_DIR,
    test_fraction=0.2,
    seed=7,
    epochs=2500,
    learning_rate=2e-3,
    residual_penalty=1e-3,
)
print("training complete")
print("overall test target metrics:")
for name, item in metrics["test_targets"].items():
    print(f"  {name}: MAE={item['mae']:.4g}, RMSE={item['rmse']:.4g}")

## 6. Save and Download Results

In [ ]:
import shutil
import zipfile

ZIP_PATH = OUTPUT_DIR / "feature_regression_results.zip"
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(OUTPUT_DIR.iterdir()):
        if path == ZIP_PATH or path.is_dir():
            continue
        zf.write(path, path.name)
print("saved:", ZIP_PATH)

try:
    from google.colab import files
    files.download(str(ZIP_PATH))
except Exception as exc:
    print("Browser download skipped or unsupported:", exc)

## 7. Optional: Mirror Results to Google Drive

In [ ]:
# Set this to True when you want Colab to mount Google Drive and copy results there.
MIRROR_TO_GOOGLE_DRIVE = False
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/Aberration-Simulation/training_results/feature_regression")

if MIRROR_TO_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    for path in OUTPUT_DIR.iterdir():
        if path.is_file():
            shutil.copy2(path, DRIVE_OUTPUT_DIR / path.name)
    print("mirrored results to:", DRIVE_OUTPUT_DIR)
else:
    print("Google Drive mirroring disabled. Set MIRROR_TO_GOOGLE_DRIVE=True and rerun this cell to copy results.")

## 8. Optional: GitHub Result Push Placeholder

In [ ]:
print("Results are saved in:", OUTPUT_DIR)
print("Automatic GitHub push from Colab is intentionally not enabled here because it requires credentials in the runtime.")
print("If needed later, add a token-based push cell after deciding how credentials should be handled.")